# Qwen3 Vigilance False-Premise Audit

vigilanceを「進む／止まる」ではなく、偽の前提を鵜呑みにするか指摘するかで再評価します。Qwen3-1.7B、source 18 → target 20、未使用20問、100 random、新seedを使用します。T4 GPUを選択してください。

In [ ]:
!nvidia-smi
!pip -q install -U "transformers>=4.53,<6" accelerate pandas

`colab_neurostate_3axis.py` と `colab_vigilance_false_premise_audit.py` を同時にアップロードしてください。重複名に `(1)` が付いても処理します。

In [ ]:
from google.colab import files
from pathlib import Path
uploaded=files.upload()
for stem in ('colab_neurostate_3axis','colab_vigilance_false_premise_audit'):
    candidates=[name for name in uploaded if name.startswith(stem) and name.endswith('.py')]
    assert candidates,f'{stem}.py がありません'
    Path(stem+'.py').write_bytes(uploaded[candidates[-1]])
print('files ready')

## 本番監査

raw vigilanceと、approach成分を除いたvigilanceを別々に評価します。

In [ ]:
!python colab_vigilance_false_premise_audit.py --random-count 100 --random-seed 20260725 --output-dir qwen3_vigilance_false_premise

In [ ]:
import json,pandas as pd
from pathlib import Path
summary=json.loads(Path('qwen3_vigilance_false_premise/summary.json').read_text())
display(pd.DataFrame(summary['results']).T[['mean_slope','positive_prompts','random_abs_beating','rank_p','max_random_abs']])
print('approach × vigilance raw cosine:',summary['approach_vigilance_raw_cosine'])
print('vigilance norm retained:',summary['vigilance_norm_retained_after_approach_removal'])
print('beating random indices:',{k:v['random_beating_indices'] for k,v in summary['results'].items()})

判定はprompt一貫性とrandom順位を優先します。approach除去後も20/20に近く、rank pが小さければ、従来の行動停止指標では見えなかったvigilance候補を支持します。これは単一token logit proxyであり、生成レベルの訂正率そのものではありません。

In [ ]:
!zip -qr qwen3_vigilance_false_premise_results.zip qwen3_vigilance_false_premise
from google.colab import files
files.download('qwen3_vigilance_false_premise_results.zip')